# Paired-coupling stochastic interpolants: schedules × solvers × ε, against DiffusionSR

Results of the `slurm/si_schedules/` experiment. Every arm is a stochastic interpolant with a **paired
(data-dependent) coupling** `x0 = m + σζ` (σ = 0.10), conditioned, like DiffusionSR, on the frozen RRDB
encoder's 64-channel features **at every network evaluation** (`si_cond: enc_implicit`).

| axis | values | trained or sampled |
|---|---|---|
| schedule | linear / flow matching · trigonometric (GVP) · VP diffusion (DDPM-matched) · Brownian bridge | trained (4) |
| paired base `m` | encoder prediction `enc(LR)` (previous task 12) · bicubic upscale (previous task 2) | trained (×2 = 8 runs) |
| solver | Euler / Euler–Maruyama · Heun / stochastic Heun · RK4 (ODE) · Dopri5 (adaptive ODE) | sampled |
| ε | 0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0 (Euler and Heun); 0 only for RK4 / Dopri5 | sampled (16 per run, 128 total) |

**Comparable with notebook 21 by construction:** same 3,829-frame test split, K = 10 samples, 20 steps,
same metric code (`evaluate_coupled.compute_metrics`), same overlay settings (fixed 293–5000 K scale, p99-clipped
error/std scales), same frozen encoder. The baselines (**DDPM in the matched harness = task 0**, **DiffusionSR as
published = task 11**, the encoder alone, and the previous SI tasks 2 and 12) are read from the previous run's own
results, not retrained.

`ε` is defined through `ε_t = ε·Γ_t²` (Γ_t the path's effective noise scale; see `runners/si_schedules.py`), so it
means the same thing for every schedule: ε = 0 is the probability-flow ODE, larger ε refreshes more of the path noise
per unit time. **Read section 11 before drawing conclusions.**

In [ ]:
# -- Configuration ----------------------------------------------------------------
import os, sys
from pathlib import Path

# ORCHARD paths. On TRACE: SI_REPO=/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom
REPO       = Path(os.getenv('SI_REPO', '/project/community/ngng/LPBFDiffusionSR_dev'))
RUNS_ROOT  = REPO / 'diffusionsr' / 'runs' / 'si_schedules'
EVAL_ROOT  = REPO / 'eval_results' / 'si_schedules'
PREV_RUNS  = REPO / 'diffusionsr' / 'runs' / 'interpolant'      # previous SI run (notebook 21)
PREV_EVAL  = REPO / 'eval_results' / 'interpolant'
CACHE      = REPO / 'eval_results' / '_predictions_cache'      # SHARED with notebook 21: no re-download
WANDB_ENTITY  = os.getenv('WANDB_ENTITY') or 'ngng-'
WANDB_PROJECT = 'Flow3D_SuperResolution'

# Identical to notebook 21, so figures compare one to one.
VMIN_T, VMAX_T = 293, 5000
MP_THRESHOLD_K = 1900.0
CH = 0

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
# -- Imports -----------------------------------------------------------------------
import json, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
# NB22_STATIC=1 renders fixed frames instead of sliders, for a pre-executed copy.
if os.getenv('NB22_STATIC'):
    HAVE_WIDGETS = False
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
# -- Arms --------------------------------------------------------------------------
SCHEDULES   = ['linear', 'trig', 'vp', 'bridge']
SCHED_LABEL = {'linear': 'linear / FM', 'trig': 'trig (GVP)', 'vp': 'VP diffusion', 'bridge': 'Brownian bridge'}
BASES       = ['enc', 'ulr']
BASE_LABEL  = {'enc': 'encoder start', 'ulr': 'bicubic start'}
SOLVERS     = ['euler', 'heun', 'rk4', 'dopri5']
SDE_SOLVERS = ['euler', 'heun']
EPS_LIST    = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

def eps_tag(e):                       # must match evaluate_si_schedules.eps_tag
    return 'eps' + f'{e:g}'.replace('.', 'p')

def new_label(sched, base, solver, eps):
    return f'{sched}·{base} | {solver} ε={eps:g}'

# Previous run, scored by the same compute_metrics on the same frames (notebook 21's arms).
PREV = [('ref', 'Encoder alone',                       'encoder_only',                      'reference'),
        (0,     'DDPM (matched harness)',              'si_baseline_ddpm_matched_temp',     'DDPM'),
        (11,    'DiffusionSR (as published)',          'si_baseline_diffusionsr_temp',      'DDPM'),
        (2,     'prev SI-A s=0.10 enc (bicubic start)', 'si_si_opt_a_sigma010_enc_temp',    'prev SI'),
        (12,    'prev SI-A s=0.10 enc, encoder start',  'si_si_opt_a_sigma010_encbase_temp', 'prev SI')]

arms = {}
for task, label, run, group in PREV:
    res = PREV_EVAL / run / f'{run}_results.json'
    if res.exists():
        arms[label] = dict(task=task, run=run, group=group, results=res, eval_dir=PREV_EVAL / run,
                           run_dir=PREV_RUNS / run, artifact=f'{run}_eval_predictions')
    else:
        print(f'previous-run result missing, skipped: [{task}] {label}')

missing = []
for sched in SCHEDULES:
    for base in BASES:
        train = f'si_sched_{sched}_{base}_temp'
        for solver in SOLVERS:
            for eps in (EPS_LIST if solver in SDE_SOLVERS else [0.0]):
                name = f'{train}__{solver}_{eps_tag(eps)}'
                res = EVAL_ROOT / name / f'{name}_results.json'
                if res.exists():
                    arms[new_label(sched, base, solver, eps)] = dict(
                        task=name, run=name, group=f'SI {sched}', results=res, eval_dir=EVAL_ROOT / name,
                        run_dir=RUNS_ROOT / train, artifact=f'{name}_eval_predictions',
                        sched=sched, base=base, solver=solver, eps=eps, train=train)
                else:
                    missing.append(name)
NEW = [l for l in arms if 'sched' in arms[l]]
print(f'{len(arms) - len(NEW)} baseline arms, {len(NEW)} of 128 new evaluations loaded'
      + (f'; {len(missing)} not evaluated yet (e.g. {missing[0]})' if missing else ''))

GROUP_COLOR = {'reference': '#555555', 'DDPM': '#cc3333', 'prev SI': '#888888',
               'SI linear': '#3366cc', 'SI trig': '#00aa77', 'SI vp': '#9467bd', 'SI bridge': '#cc8800'}
SCHED_COLOR = {s: GROUP_COLOR[f'SI {s}'] for s in SCHEDULES}
SOLVER_MARKER = {'euler': 'o', 'heun': 's', 'rk4': '^', 'dopri5': 'D'}

## 1 - Metrics

Same definitions as notebook 21 (`compute_metrics()` in `analysis/evaluate_coupled.py`, physical units, K = 10):
**MAE / RMSE** per sample; **ensemble MAE** of the K-sample mean; **MP / KH MAE** melt-pool and keyhole-depression
depth error (µm, 1900 K region); **area MAE** (µm²); **calibration_2sigma**, ideal ≈ **0.95**.
New columns: **spread (melt)**, the mean per-pixel ensemble std inside the true melt pool (K), the number notebook 21's
section 6 reports; **NFE**, network evaluations per sample (DDIM 20 steps = 20; Heun 20 steps = 41).

Prediction arrays are only needed for the 5 baseline arms' spread (from notebook 21's shared cache) and for the
figures in sections 7-10; every other number comes from `results.json`.

In [ ]:
# -- Prediction arrays: notebook 21's loader, plus the artifact name ----------------
def cached_arrays(run, eval_dir, artifact):
    d = CACHE / run
    if not ((d / 'pred.npy').exists() and (d / 'gt.npy').exists()):
        d.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=CACHE, ignore_cleanup_errors=True) as tmp:
            src = eval_dir / 'predictions.npz'
            if not src.exists():
                import wandb
                wandb.Api().artifact(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{artifact}:latest').download(root=tmp)
                src = Path(tmp) / 'predictions.npz'
            with np.load(src, allow_pickle=True) as z:
                np.save(d / 'pred.npy', z['pred'])
                np.save(d / 'gt.npy', z['gt'])
    return np.load(d / 'pred.npy', mmap_mode='r'), np.load(d / 'gt.npy', mmap_mode='r')

CACHE.mkdir(parents=True, exist_ok=True)
pred, gt = {}, None

def get_pred(label):
    """Load one arm's (N, K, C, H, W) predictions on first use; check every arm scores the same frames."""
    global gt
    if label not in pred:
        a = arms[label]
        p, g = cached_arrays(a['run'], a['eval_dir'], a['artifact'])
        if gt is None:
            gt = g
        elif g.shape != gt.shape or not np.array_equal(g[::500], gt[::500]):
            raise RuntimeError(f'{label}: ground truth differs from the other arms -- not the same test frames')
        pred[label] = p
    return pred[label]

def spread_of(p, g):
    """Notebook 21 section 6 / evaluate_si_schedules.ensemble_spread, streamed in chunks."""
    tot = tot_mp = 0.0; n_mp = 0
    for s in range(0, p.shape[0], 256):
        sd = np.asarray(p[s:s + 256, :, CH], dtype=np.float32).std(axis=1)
        mask = np.asarray(g[s:s + 256, CH]) > MP_THRESHOLD_K
        tot += float(sd.sum()); tot_mp += float(sd[mask].sum()); n_mp += int(mask.sum())
    return tot / sd[0].size / p.shape[0], tot_mp / max(n_mp, 1)

In [ ]:
# -- Metrics table -----------------------------------------------------------------
SHOW = ['MAE_mean', 'MAE_std', 'MAE_ensemble_mean', 'RMSE_mean',
        'MP_MAE_mean', 'KH_MAE_mean', 'area_mae_um2', 'calibration_2sigma']
EXTRA = ['spread_melt_px_K', 'spread_all_px_K', 'nfe_per_sample']
rows = {}
for label, a in arms.items():
    r = json.load(open(a['results']))
    row = {'task': a['task'] if 'sched' not in a else '', 'group': a['group'],
           'schedule': a.get('sched', ''), 'base': a.get('base', ''), 'solver': a.get('solver', ''),
           'eps': a.get('eps', np.nan), **{k: r.get(k, np.nan) for k in SHOW + EXTRA}}
    if a['group'] == 'reference':
        row['calibration_2sigma'] = np.nan          # K = 1: no ensemble to be calibrated
    if a['group'] in ('DDPM', 'prev SI', 'reference'):
        row['nfe_per_sample'] = {'reference': 1, 'DDPM': 20, 'prev SI': 41}[a['group']]   # DDIM 20 / Heun 20
        if a['group'] != 'reference':
            try:
                row['spread_all_px_K'], row['spread_melt_px_K'] = spread_of(get_pred(label), gt)
            except Exception as e:
                print(f'no predictions for {label} (spread left blank): {str(e).splitlines()[0][:100]}')
    rows[label] = row
metrics = pd.DataFrame(rows).T
for k in SHOW + EXTRA + ['eps']:
    metrics[k] = pd.to_numeric(metrics[k], errors='coerce')
metrics.to_csv(EVAL_ROOT / 'all_results.csv') if EVAL_ROOT.exists() else None

def _best(df, col):
    s = df[col].dropna()
    if s.empty:
        return None
    return (s - 0.95).abs().idxmin() if col == 'calibration_2sigma' else s.idxmin()

def show_table(df, cols=SHOW + ['spread_melt_px_K', 'nfe_per_sample']):
    best = {c: _best(df, c) for c in cols if c not in ('MAE_std', 'spread_melt_px_K', 'nfe_per_sample')}
    hl = lambda col: ['font-weight: bold; color: #0a7' if best.get(col.name) == i else '' for i in col.index]
    display(df[['group'] + cols].style.format({k: '{:.4g}' for k in cols}).apply(hl, axis=0, subset=cols))

BASELINES = [l for l in arms if 'sched' not in arms[l]]
# The previous run's exact protocol: deterministic Heun, 20 steps. One row per (schedule, base).
MATCHED = [new_label(s, b, 'heun', 0.0) for b in BASES for s in SCHEDULES if new_label(s, b, 'heun', 0.0) in arms]
print('Baselines and the matched-protocol point (Heun, eps = 0, 20 steps) of every new run. Bold = best per column.')
show_table(metrics.loc[BASELINES + MATCHED])

In [ ]:
# -- Best configuration per (schedule, base) for each headline metric --------------
HEAD = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MP_MAE_mean', 'MP depth MAE (um)'),
        ('KH_MAE_mean', 'KH depth MAE (um)'), ('area_mae_um2', 'area MAE (um^2)'),
        ('calibration_2sigma', 'calibration (~0.95)')]
best_rows = []
for s in SCHEDULES:
    for b in BASES:
        sub = metrics[(metrics.schedule == s) & (metrics.base == b)]
        if sub.empty:
            continue
        row = {'schedule': SCHED_LABEL[s], 'base': BASE_LABEL[b]}
        for m, name in HEAD:
            i = _best(sub, m)
            row[name] = f"{sub.loc[i, m]:.4g}  ({sub.loc[i, 'solver']}, ε={sub.loc[i, 'eps']:g})" if i else ''
        best_rows.append(row)
ref = {name: f"{metrics.loc['DDPM (matched harness)', m]:.4g}" for m, name in HEAD} if 'DDPM (matched harness)' in metrics.index else {}
print('Best solver/eps per run and metric. DDPM (task 0) for reference:', ref)
display(pd.DataFrame(best_rows))

print('\nAll evaluations (sortable copy saved as eval_results/si_schedules/all_results.csv):')
show_table(metrics.loc[NEW].sort_values(['schedule', 'base', 'solver', 'eps']))

## 2 - The questions the design can answer, one change at a time

Each pair differs in exactly one factor. The first two pairs are **reproduction checks** against the previous run:
same schedule, start, solver and step count; they differ only in the network now having a second (denoiser) head and the
velocity being assembled as `v̂ + Γ̇·η̂` instead of regressed directly. A large gap there would point at the
parametrisation, not at the schedule.

In [ ]:
# -- One-factor comparisons ----------------------------------------------------------
H = lambda s, b, sv='heun', e=0.0: new_label(s, b, sv, e)
PAIRS = [('Reproduction: previous task 12 -> linear·enc (Heun, eps=0)', 'prev SI-A s=0.10 enc, encoder start', H('linear', 'enc')),
         ('Reproduction: previous task 2 -> linear·ulr (Heun, eps=0)', 'prev SI-A s=0.10 enc (bicubic start)', H('linear', 'ulr'))]
PAIRS += [(f'Schedule: linear -> {s} (enc, Heun, eps=0)', H('linear', 'enc'), H(s, 'enc')) for s in SCHEDULES[1:]]
PAIRS += [(f'Start: encoder -> bicubic ({s}, Heun, eps=0)', H(s, 'enc'), H(s, 'ulr')) for s in SCHEDULES]
PAIRS += [(f'Solver: heun -> {sv} (linear·enc, eps=0)', H('linear', 'enc'), H('linear', 'enc', sv)) for sv in ['euler', 'rk4', 'dopri5']]
PAIRS += [(f'Sampling noise: eps 0 -> {e:g} (linear·enc, Heun)', H('linear', 'enc'), H('linear', 'enc', 'heun', e)) for e in [0.1, 1.0]]
if 'DDPM (matched harness)' in metrics.index and NEW:
    best_new = metrics.loc[NEW, 'MAE_ensemble_mean'].idxmin()
    PAIRS += [(f'DDPM (task 0) -> best new arm by ensemble MAE: {best_new}', 'DDPM (matched harness)', best_new),
              ('DiffusionSR as published (task 11) -> DDPM in the harness (task 0)', 'DiffusionSR (as published)', 'DDPM (matched harness)')]
CMP = ['MAE_ensemble_mean', 'MP_MAE_mean', 'KH_MAE_mean', 'area_mae_um2', 'calibration_2sigma', 'spread_melt_px_K']
out = []
for q, a, b in PAIRS:
    if a not in metrics.index or b not in metrics.index:
        continue
    for m in CMP:
        va, vb = metrics.loc[a, m], metrics.loc[b, m]
        if m == 'calibration_2sigma':
            verdict = 'better' if abs(vb - 0.95) < abs(va - 0.95) else 'worse'
        elif m == 'spread_melt_px_K':
            verdict = 'more spread' if vb > va else 'less spread'
        else:
            verdict = 'better' if vb < va else 'worse'
        out.append({'question': q, 'metric': m, 'from': va, 'to': vb,
                    'change': 100.0 * (vb - va) / va if va else np.nan, 'verdict': verdict})
display(pd.DataFrame(out).style.format({'from': '{:.4g}', 'to': '{:.4g}', 'change': '{:+.1f}%'}).hide(axis='index'))

## 3 - The ε sweep

Euler–Maruyama and stochastic Heun at every ε, per schedule (columns) and metric (rows). Solid = encoder start,
dashed = bicubic start; circles = Euler, squares = Heun. Horizontal lines: DDPM in the matched harness (task 0, red) and
DiffusionSR as published (task 11, red dotted); grey dotted on the calibration row = the ideal 0.95.
The x-axis is symmetric-log so that ε = 0 is shown.

In [ ]:
# -- Metric vs eps ---------------------------------------------------------------------
SWEEP = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MP_MAE_mean', 'MP depth MAE (um)'),
         ('KH_MAE_mean', 'KH depth MAE (um)'), ('area_mae_um2', 'area MAE (um^2)'),
         ('calibration_2sigma', 'calibration'), ('spread_melt_px_K', 'spread in melt pool (K)')]
fig, axes = plt.subplots(len(SWEEP), len(SCHEDULES), figsize=(4.3 * len(SCHEDULES), 2.6 * len(SWEEP)),
                         sharex=True, squeeze=False)
for j, s in enumerate(SCHEDULES):
    for i, (m, name) in enumerate(SWEEP):
        ax = axes[i, j]
        for b, ls in [('enc', '-'), ('ulr', '--')]:
            for sv in SDE_SOLVERS:
                sub = metrics[(metrics.schedule == s) & (metrics.base == b) & (metrics.solver == sv)].sort_values('eps')
                if len(sub):
                    ax.plot(sub.eps, sub[m], ls, marker=SOLVER_MARKER[sv], ms=4, color=SCHED_COLOR[s],
                            alpha=1.0 if sv == 'heun' else 0.55, label=f'{BASE_LABEL[b]}, {sv}')
        for lab, sty in [('DDPM (matched harness)', '-'), ('DiffusionSR (as published)', ':')]:
            if lab in metrics.index and np.isfinite(metrics.loc[lab, m]):
                ax.axhline(metrics.loc[lab, m], color='#cc3333', ls=sty, lw=1)
        if m == 'calibration_2sigma':
            ax.axhline(0.95, color='grey', ls=':', lw=1)
        ax.set_xscale('symlog', linthresh=0.01)
        ax.grid(alpha=0.3)
        if i == 0:
            ax.set_title(SCHED_LABEL[s], fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=8)
        if i == len(SWEEP) - 1:
            ax.set_xlabel('ε')
handles, labels = axes[0, 0].get_legend_handles_labels()
handles += [plt.Line2D([], [], color='#cc3333', lw=1), plt.Line2D([], [], color='#cc3333', ls=':', lw=1)]
labels += ['DDPM, matched harness (task 0)', 'DiffusionSR as published (task 11)']
fig.legend(handles, labels, loc='lower center', ncol=6, fontsize=8, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 4 - Solvers at ε = 0, and quality per unit compute

Left: every solver for every run at ε = 0 (the ODE), with DDPM (task 0) as the red line. Right: ensemble MAE against
network evaluations per sample, the compute that actually decides sampling cost. At a fixed 20 steps, RK4 spends 4× and
Heun 2× the evaluations of Euler; Dopri5 picks its own.

In [ ]:
# -- Solver comparison at eps = 0 -----------------------------------------------------
ode = metrics.loc[NEW][metrics.loc[NEW, 'eps'] == 0.0]
runs = [(s, b) for b in BASES for s in SCHEDULES]
fig, axes = plt.subplots(1, 3, figsize=(19, 4.2), gridspec_kw={'width_ratios': [1.4, 1.4, 1]})
for ax, (m, name) in zip(axes[:2], [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('KH_MAE_mean', 'KH depth MAE (um)')]):
    w = 0.2
    for k, sv in enumerate(SOLVERS):
        vals = [ode[(ode.schedule == s) & (ode.base == b) & (ode.solver == sv)][m].mean() for s, b in runs]
        ax.bar(np.arange(len(runs)) + (k - 1.5) * w, vals, w, label=sv, color=plt.cm.Greys(0.35 + 0.18 * k))
    if 'DDPM (matched harness)' in metrics.index:
        ax.axhline(metrics.loc['DDPM (matched harness)', m], color='#cc3333', lw=1.2, label='DDPM (task 0)')
    ax.set_xticks(range(len(runs)))
    ax.set_xticklabels([f'{s}\n{b}' for s, b in runs], fontsize=8)
    ax.set_title(f'{name}, eps = 0', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
axes[0].legend(fontsize=8)
ax = axes[2]
for s in SCHEDULES:
    for sv in SOLVERS:
        sub = ode[(ode.schedule == s) & (ode.solver == sv)]
        ax.scatter(sub.nfe_per_sample, sub.MAE_ensemble_mean, color=SCHED_COLOR[s], marker=SOLVER_MARKER[sv], s=30,
                   facecolors=[SCHED_COLOR[s] if b == 'enc' else 'none' for b in sub.base])
for lab, mk in [('DDPM (matched harness)', '*'), ('DiffusionSR (as published)', 'P')]:
    if lab in metrics.index:
        ax.scatter(metrics.loc[lab, 'nfe_per_sample'], metrics.loc[lab, 'MAE_ensemble_mean'], color='#cc3333', marker=mk, s=120)
ax.set_xscale('log'); ax.set_xlabel('network evaluations per sample'); ax.set_ylabel('ensemble MAE (K)')
ax.set_title('quality vs compute (filled = encoder start)', fontsize=10); ax.grid(alpha=0.3)
hs = ([plt.Line2D([], [], ls='', marker='o', color=c, label=SCHED_LABEL[s]) for s, c in SCHED_COLOR.items()]
      + [plt.Line2D([], [], ls='', marker=mk, color='k', label=sv) for sv, mk in SOLVER_MARKER.items()]
      + [plt.Line2D([], [], ls='', marker='*', color='#cc3333', ms=10, label='DDPM (task 0)')])
ax.legend(handles=hs, fontsize=7)
plt.tight_layout(); plt.show()

## 5 - Fidelity versus diversity

The previous run's central finding was that the interpolant **under-disperses**: calibration 0.63–0.65 against DDPM's
0.85, with about half DDPM's spread in the melt pool. ε exists to buy that diversity back. Each line follows one
(run, solver) as ε rises; the best place to be is **low error at DDPM-like spread and calibration near 0.95**.

In [ ]:
# -- Frontier: spread / calibration vs error, traced along eps -----------------------
fig, axes = plt.subplots(2, len(SCHEDULES), figsize=(4.4 * len(SCHEDULES), 8), squeeze=False)
for j, s in enumerate(SCHEDULES):
    for i, (xm, xname) in enumerate([('spread_melt_px_K', 'spread in melt pool (K)'), ('calibration_2sigma', 'calibration')]):
        ax = axes[i, j]
        for b, ls in [('enc', '-'), ('ulr', '--')]:
            for sv in SDE_SOLVERS:
                sub = metrics[(metrics.schedule == s) & (metrics.base == b) & (metrics.solver == sv)].sort_values('eps')
                if len(sub):
                    ax.plot(sub[xm], sub.MAE_ensemble_mean, ls, marker=SOLVER_MARKER[sv], ms=4, color=SCHED_COLOR[s],
                            alpha=1.0 if sv == 'heun' else 0.55)
                    ax.annotate(f"ε={sub.eps.iloc[-1]:g}", (sub[xm].iloc[-1], sub.MAE_ensemble_mean.iloc[-1]), fontsize=6)
        for lab, mk in [('DDPM (matched harness)', '*'), ('DiffusionSR (as published)', 'P'),
                        ('prev SI-A s=0.10 enc, encoder start', 'X')]:
            if lab in metrics.index and np.isfinite(metrics.loc[lab, xm]):
                ax.scatter(metrics.loc[lab, xm], metrics.loc[lab, 'MAE_ensemble_mean'],
                           color='#cc3333' if 'prev' not in lab else '#888888', marker=mk, s=110, zorder=5, label=lab)
        if xm == 'calibration_2sigma':
            ax.axvline(0.95, color='grey', ls=':', lw=1)
        ax.set_xlabel(xname); ax.grid(alpha=0.3)
        if j == 0:
            ax.set_ylabel('ensemble MAE (K)')
        if i == 0:
            ax.set_title(SCHED_LABEL[s], fontsize=10)
axes[0, 0].legend(fontsize=7)
plt.suptitle('solid = encoder start, dashed = bicubic start; squares = Heun, circles = Euler; points run eps = 0 -> 1', fontsize=9)
plt.tight_layout(); plt.show()

## 6 - Training

One run per (schedule, base). Loss values are comparable **within a schedule** (enc vs bicubic) but only loosely across
schedules: every arm regresses the same two heads, but the size of the velocity target depends on the schedule.
Dots mark the best (lowest validation) epoch, which is the checkpoint every eval used. The previous run's losses
(single-head velocity regression) and DDPM's (ε-prediction) are on different objectives and are not plotted here.

In [ ]:
# -- Loss curves -----------------------------------------------------------------------
fig, axes = plt.subplots(2, len(SCHEDULES), figsize=(4.6 * len(SCHEDULES), 7), squeeze=False)
summary = {}
for j, s in enumerate(SCHEDULES):
    for b, ls in [('enc', '-'), ('ulr', '--')]:
        d = RUNS_ROOT / f'si_sched_{s}_{b}_temp'
        for i, fname in enumerate(['loss_epoch.txt', 'validation_loss_epoch.txt']):
            f = d / fname
            if not f.exists():
                continue
            y = np.loadtxt(f).reshape(-1)
            line, = axes[i, j].plot(y, ls, lw=1, color=SCHED_COLOR[s], label=BASE_LABEL[b])
            if i == 1:
                k = int(np.argmin(y))
                axes[i, j].plot(k, y[k], 'o', ms=4, color=line.get_color())
                summary[f'{s}·{b}'] = {'epochs': len(y), 'best epoch': k, 'best val loss': y[k],
                                       'final val loss': y[-1], 'final / best': y[-1] / y[k]}
    for i, kind in enumerate(['train', 'validation']):
        ax = axes[i, j]
        ax.set_yscale('log'); ax.set_title(f'{SCHED_LABEL[s]}: {kind}', fontsize=10); ax.set_xlabel('epoch')
        if ax.lines:
            ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
if summary:
    display(pd.DataFrame(summary).T.style.format({'best val loss': '{:.4g}', 'final val loss': '{:.4g}', 'final / best': '{:.2f}'}))

## 7 - Overlays

Notebook 21's overlay, unchanged: one shared frame, the fixed 293–5000 K scale, columns = ground truth, three samples,
ensemble mean, |mean − GT| and per-pixel ensemble std (both on one scale shared by every row, clipped at the 99th
percentile). The default frame is the one with the largest melt pool, as in notebook 21.
Edit `SHOW_ARMS` (any label from the metrics table) to change the rows; arrays are fetched on first use.

In [ ]:
# -- Overlay arms and frame selection --------------------------------------------------
SHOW_ARMS = (['Encoder alone', 'DDPM (matched harness)', 'DiffusionSR (as published)', 'prev SI-A s=0.10 enc, encoder start']
             + [new_label(s, 'enc', 'heun', 0.0) for s in SCHEDULES])
SHOW_ARMS = [l for l in SHOW_ARMS if l in arms]
for l in SHOW_ARMS:
    try:
        get_pred(l)
    except Exception as e:
        print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')

N_FRAMES = 0 if gt is None else gt.shape[0]
HOT = (np.asarray(gt[:, CH]) > MP_THRESHOLD_K).sum(axis=(1, 2)) if N_FRAMES else np.array([])
DEFAULT_FRAME = int(np.argmax(HOT)) if N_FRAMES else 0
print(f'{N_FRAMES} test frames, {int((HOT > 0).sum())} with a melt pool; default frame {DEFAULT_FRAME}')

def with_frame_slider(render_fn, default=DEFAULT_FRAME):
    if N_FRAMES == 0:
        return
    if not HAVE_WIDGETS:
        render_fn(default)
        return
    slider = widgets.IntSlider(min=0, max=N_FRAMES - 1, value=default, description='frame:',
                               continuous_update=False, layout=widgets.Layout(width='460px'))
    display(widgets.VBox([slider, widgets.interactive_output(render_fn, {'frame': slider})]))

def row_label(l):
    a = arms[l]
    return f"[{a['task']}] {l}" if 'sched' not in a else l

In [ ]:
# -- Rebuilt overlay: shared frame, shared scales (notebook 21 section 4) -------------
N_SAMPLES_SHOWN = 3

def render_overlay(frame, show=None):
    labels = [l for l in (show or SHOW_ARMS) if l in pred]
    if not labels:
        return
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in labels}
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in labels]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in labels]), 99)) or 1.0
    ncol = 1 + N_SAMPLES_SHOWN + 3
    fig, axes = plt.subplots(len(labels), ncol, figsize=(2.2 * ncol, 2.2 * len(labels)), squeeze=False)
    col_titles = (['ground truth'] + [f'sample {k + 1}' for k in range(N_SAMPLES_SHOWN)]
                  + ['ensemble mean', f'|mean - GT|, 0-{err_max:.0f} K (p99)', f'ensemble std, 0-{std_max:.0f} K (p99)'])
    for r, l in enumerate(labels):
        p, m = P[l], P[l].mean(0)
        panels = [(g, 'jet', VMIN_T, VMAX_T)]
        panels += [(p[k], 'jet', VMIN_T, VMAX_T) if k < len(p) else None for k in range(N_SAMPLES_SHOWN)]
        panels += [(m, 'jet', VMIN_T, VMAX_T), (np.abs(m - g), 'magma', 0, err_max), (p.std(0), 'hot', 0, std_max)]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=8)
            if panel is None:
                ax.axis('off'); continue
            img, cmap, vmin, vmax = panel
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        axes[r, 0].set_ylabel(row_label(l), fontsize=7)
    fig.suptitle(f'Frame {frame} - temperature (K), fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout(); plt.show()

with_frame_slider(render_overlay)

In [ ]:
# -- What eps does to the samples: one run, Heun, every eps (same overlay settings) ---
EPS_RUN = ('linear', 'enc')          # edit to inspect another (schedule, base)
EPS_ROWS = [new_label(*EPS_RUN, 'heun', e) for e in EPS_LIST if new_label(*EPS_RUN, 'heun', e) in arms]
for l in EPS_ROWS:
    try:
        get_pred(l)
    except Exception as e:
        print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')
with_frame_slider(lambda frame: render_overlay(frame, show=['DDPM (matched harness)'] + EPS_ROWS))

In [ ]:
# -- The overlay PNG each eval rendered itself (frame 70, per-run colour scale: do NOT compare by eye) --
for l in SHOW_ARMS:
    png = arms[l]['eval_dir'] / 'overlay_frame70.png'
    if png.exists():
        print(row_label(l))
        display(Image(filename=str(png), width=1100))

## 8 - Melt-pool boundaries

Notebook 21 section 5, unchanged: the 1900 K isotherm of each arm's ensemble mean against the ground truth (black), and
every individual sample's boundary for the arms in `SPREAD_ARMS`.

In [ ]:
# -- Melt-pool contours ----------------------------------------------------------------
PALETTE = ['#00d5ff', '#cc3333', '#ff7f0e', '#3366cc', '#9467bd', '#00aa77', '#cc8800', '#17becf']
SPREAD_ARMS = ['DDPM (matched harness)', new_label('linear', 'enc', 'heun', 0.0), new_label('linear', 'enc', 'heun', 1.0)]
for l in SPREAD_ARMS:
    if l in arms:
        try:
            get_pred(l)
        except Exception as e:
            print(f'no predictions for {l}: {str(e).splitlines()[0][:120]}')

def _crop(ax, g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)
    if len(hot):
        (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
        ax.set_xlim(max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5)
        ax.set_ylim(max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

def render_contours(frame):
    g = np.asarray(gt[frame, CH])
    spread = [l for l in SPREAD_ARMS if l in pred]
    fig, axes = plt.subplots(1, 1 + len(spread), figsize=(6 * (1 + len(spread)), 5.5), squeeze=False)
    axes = axes[0]
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        _crop(ax, g)
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for i, l in enumerate([x for x in SHOW_ARMS if x in pred]):
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        col = PALETTE[i % len(PALETTE)]
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[col], linewidths=1.3)
        handles.append(plt.Line2D([], [], color=col, label=row_label(l)))
    axes[0].legend(handles=handles, fontsize=7, loc='lower left')
    axes[0].set_title(f'Frame {frame}: 1900 K boundary of each ensemble mean', fontsize=10)
    for ax, l in zip(axes[1:], spread):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=['#cc3333'], linewidths=0.8, alpha=0.7)
        ax.set_title(f'{row_label(l)}: all {len(p)} samples (red)', fontsize=9)
    plt.tight_layout(); plt.show()

with_frame_slider(render_contours)

## 9 - Ensemble spread

Notebook 21 section 6: per-pixel ensemble std maps on one shared scale (clipped at p99), and the test-set averages,
overall and inside the true melt pool, next to calibration (ideal ≈ 0.95). For new arms these averages come from the
eval itself (`spread_*_px_K` in `results.json`, same formula).

In [ ]:
# -- Per-pixel ensemble std, same slider ----------------------------------------------
ENS_ARMS = [l for l in SHOW_ARMS if l in pred and pred[l].shape[1] > 1]

def render_std(frame):
    S = {l: np.asarray(pred[l][frame, :, CH]).std(0) for l in ENS_ARMS}
    if not S:
        return
    vmax = float(np.percentile(np.stack(list(S.values())), 99)) or 1.0
    fig, axes = plt.subplots(1, len(S), figsize=(3.1 * len(S), 3.3), squeeze=False)
    for ax, (l, s) in zip(axes[0], S.items()):
        im = ax.imshow(s.T, origin='lower', cmap='hot', vmin=0, vmax=vmax)
        ax.set_title(row_label(l), fontsize=7); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[0].tolist(), fraction=0.02, label='std (K)')
    fig.suptitle(f'Frame {frame}: per-pixel ensemble std, shared scale 0-{vmax:.0f} K (clipped at p99)', fontsize=10)
    plt.show()

with_frame_slider(render_std)

cols = ['spread_all_px_K', 'spread_melt_px_K', 'calibration_2sigma', 'MAE_ensemble_mean']
tab = metrics.loc[[l for l in BASELINES if arms[l]['group'] != 'reference'] + MATCHED, cols]
tab.columns = ['mean std, all pixels (K)', 'mean std, melt-pool pixels (K)', 'calibration_2sigma', 'ensemble MAE (K)']
display(tab.style.format('{:.3g}'))

## 10 - Head-to-head: DiffusionSR vs the best new configuration

Notebook 21 section 8's layout, for DDPM in the matched harness (task 0), DiffusionSR as published (task 11), the
previous best interpolant (task 12) and the best new configuration by ensemble MAE: all 10 samples per arm, ensemble
mean, error and spread, cropped to the melt region, for the largest, a median and a small melt pool.

In [ ]:
# -- Head-to-head metrics ----------------------------------------------------------------
BEST_NEW = metrics.loc[NEW, 'MAE_ensemble_mean'].idxmin() if NEW else None
H2H = [l for l in ['DDPM (matched harness)', 'DiffusionSR (as published)', 'prev SI-A s=0.10 enc, encoder start', BEST_NEW]
       if l in arms]
H2H_COLOR = dict(zip(H2H, ['#cc3333', '#ff7f0e', '#888888', '#3366cc']))
for l in H2H:
    get_pred(l)
H2H_METRICS = [('MAE_ensemble_mean', 'ensemble MAE (K)'), ('MAE_mean', 'per-sample MAE (K)'), ('RMSE_mean', 'RMSE (K)'),
               ('MP_MAE_mean', 'melt-pool depth MAE (um)'), ('KH_MAE_mean', 'keyhole depth MAE (um)'),
               ('area_mae_um2', 'melt-pool area MAE (um^2)'), ('calibration_2sigma', 'calibration (ideal ~0.95)'),
               ('spread_melt_px_K', 'sample spread in melt pool (K)'), ('nfe_per_sample', 'network evals / sample')]
base = H2H[0]
table = {}
for l in H2H:
    col = {}
    for m, name in H2H_METRICS:
        v, v0 = metrics.loc[l, m], metrics.loc[base, m]
        col[name] = f'{v:.4g}' if l == base else f'{v:.4g}  ({100.0 * (v - v0) / v0:+.0f}%)'
    table[row_label(l)] = col
print('Brackets: change relative to task 0. Lower is better for every error row; calibration is best near 0.95.')
display(pd.DataFrame(table))

melted = np.where(HOT > 0)[0]
by_size = melted[np.argsort(HOT[melted], kind='stable')]
H2H_FRAMES = {'largest melt pool': int(by_size[-1]), 'median melt pool': int(by_size[len(by_size) // 2]),
              'small melt pool': int(by_size[len(by_size) // 10])}

def show_frames(render_fn):
    if HAVE_WIDGETS:
        with_frame_slider(render_fn)
    else:
        for f in H2H_FRAMES.values():
            render_fn(f)

def _melt_box(g, margin=6):
    hot = np.argwhere(g > MP_THRESHOLD_K)
    if not len(hot):
        return None
    (x0, z0), (x1, z1) = hot.min(0), hot.max(0)
    return (max(x0 - margin, 0) - 0.5, min(x1 + margin, g.shape[0] - 1) + 0.5,
            max(z0 - margin, 0) - 0.5, min(z1 + margin, g.shape[1] - 1) + 0.5)

def render_h2h_overlay(frame):
    g = np.asarray(gt[frame, CH])
    P = {l: np.asarray(pred[l][frame, :, CH]) for l in H2H}
    K = max(p.shape[0] for p in P.values())
    err_max = float(np.percentile(np.stack([np.abs(P[l].mean(0) - g) for l in H2H]), 99)) or 1.0
    std_max = float(np.percentile(np.stack([P[l].std(0) for l in H2H]), 99)) or 1.0
    box = _melt_box(g)
    titles = (['ground truth'] + [f'sample {k + 1}' for k in range(K)]
              + ['ensemble mean', f'|mean - GT|\n0-{err_max:.0f} K (p99)', f'ensemble std\n0-{std_max:.0f} K (p99)'])
    fig, axes = plt.subplots(len(H2H), len(titles), figsize=(1.55 * len(titles), 1.9 * len(H2H) + 0.6), squeeze=False)
    for r, l in enumerate(H2H):
        p, m = P[l], P[l].mean(0)
        imgs = ([(g, 'jet', VMIN_T, VMAX_T)] + [(p[k], 'jet', VMIN_T, VMAX_T) for k in range(len(p))]
                + [None] * (K - len(p))
                + [(m, 'jet', VMIN_T, VMAX_T), (np.abs(m - g), 'magma', 0, err_max), (p.std(0), 'hot', 0, std_max)])
        for c, panel in enumerate(imgs):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(titles[c], fontsize=7)
            if panel is None:
                ax.axis('off'); continue
            img, cmap, vmin, vmax = panel
            ax.imshow(img.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
            if box:
                ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
        axes[r, 0].set_ylabel(row_label(l), fontsize=6, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): every sample, fixed scale {VMIN_T}-{VMAX_T} K', fontsize=10)
    plt.tight_layout(); plt.show()

show_frames(render_h2h_overlay)

In [ ]:
# -- Head-to-head melt-pool boundaries -------------------------------------------------
def render_h2h_contours(frame):
    g = np.asarray(gt[frame, CH])
    box = _melt_box(g)
    fig, axes = plt.subplots(1, 1 + len(H2H), figsize=(4.6 * (1 + len(H2H)), 4.6))
    for ax in axes:
        ax.imshow(g.T, origin='lower', cmap='Greys_r', vmin=VMIN_T, vmax=VMAX_T)
        if g.max() > MP_THRESHOLD_K:
            ax.contour(g.T, levels=[MP_THRESHOLD_K], colors='k', linewidths=2.5)
        if box:
            ax.set_xlim(box[0], box[1]); ax.set_ylim(box[2], box[3])
    handles = [plt.Line2D([], [], color='k', lw=2.5, label='ground truth')]
    for l in H2H:
        m = np.asarray(pred[l][frame, :, CH]).mean(0)
        if m.max() > MP_THRESHOLD_K:
            axes[0].contour(m.T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=1.6)
        handles.append(plt.Line2D([], [], color=H2H_COLOR[l], lw=1.6, label=row_label(l)))
    axes[0].legend(handles=handles, fontsize=6, loc='lower left')
    axes[0].set_title('ensemble-mean boundary (1900 K)', fontsize=10)
    for ax, l in zip(axes[1:], H2H):
        p = np.asarray(pred[l][frame, :, CH])
        for k in range(len(p)):
            if p[k].max() > MP_THRESHOLD_K:
                ax.contour(p[k].T, levels=[MP_THRESHOLD_K], colors=[H2H_COLOR[l]], linewidths=0.8, alpha=0.75)
        ax.set_title(f'all {len(p)} sample boundaries', fontsize=9, color=H2H_COLOR[l])
    name = next((k for k, f in H2H_FRAMES.items() if f == frame), 'selected frame')
    fig.suptitle(f'Frame {frame} ({name}): black = ground truth', fontsize=10)
    plt.tight_layout(); plt.show()

show_frames(render_h2h_contours)

## 11 - Reading this correctly

- **What is held equal.** Test frames, K = 10, the 20-step budget, metric code, the frozen encoder and its per-step
  conditioning, data, harness (EMA, grad clip, StepLR, flips) and every training hyperparameter. Within the new study,
  `(schedule, base)` are the only trained variables; solver and ε are sampling-only and share one checkpoint.
- **Matched-protocol rows** (Heun, ε = 0, 20 steps) are the ones directly comparable with notebook 21's interpolant
  arms. Sections 3-5 then show what the rest of the grid buys.
- **Step count vs compute.** "20 steps" means 21 network evaluations for Euler, 41 for Heun, 81 for RK4 and a variable
  number for Dopri5 (DDIM: 20). Use the NFE column / section 4 before calling one solver better than another.
- **Euler on the bridge and VP schedules** has a real first-order bias near t → 1, where those schedules' coefficients
  are singular (tested on a Gaussian: std 1.50 / 0.80 / 0.71 at 20 / 200 / 2000 steps against 0.70). That is a property
  of the (solver, schedule) pair, not a bug.
- **ε definition.** ε_t = ε·Γ_t², so ε has the same meaning for every schedule; it is not the previous run's `eps0`
  (which was only ever defined for Option B and was never swept).
- **Not a reproduction of task 12 to the digit.** The linear/encoder arm differs from task 12 in having a second
  (denoiser) head and assembling the velocity as v̂ + Γ̇η̂; section 2's reproduction rows show how much that moves.
- **Checkpoint used.** Every eval loads `bestmodel_saved.pth`'s raw weights (not the EMA copy), exactly as the previous
  evals did, so the two studies are scored the same way. `--weights ema` re-scores with the EMA weights if wanted.
- **Physics metrics in µm assume 10 µm per HR pixel**, as in notebook 21; rankings do not depend on it.